# Feature Importance Comparison (High-Low Indicators)

This notebook compares the feature importance of the stock-ai-quant pipeline before and after adding the new High-Low technical indicators.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib

from core.data_loader import fetch_market_data, fetch_macro_data, fetch_fundamentals_data
from core.features import build_features_and_target
from core.model import train_model_and_evaluate

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
ticker = 'AAPL'

# Load basic data
df_stock = fetch_market_data(ticker, period='2y')
df_sp500, df_usdjpy, df_nikkei, df_tnx = fetch_macro_data(period='2y')
df_fund = fetch_fundamentals_data(ticker)

# Create dummy sentiment and attention data for isolation
df_daily_sentiment = pd.DataFrame(index=df_stock.index)
df_daily_sentiment['Sentiment_Score'] = 0.0
df_daily_sentiment['Date'] = df_daily_sentiment.index

df_attention = pd.Series(0.0, index=df_stock.index)
df_jobs = pd.Series(0.0, index=df_stock.index)

In [ ]:
def train_and_get_importance(df_stock_input):
    clean_df, latest_df, feature_cols = build_features_and_target(
        df_stock=df_stock_input,
        df_sp500=df_sp500,
        df_usdjpy=df_usdjpy,
        df_nikkei=df_nikkei,
        df_daily_sentiment=df_daily_sentiment,
        df_fund=df_fund,
        ticker=ticker,
        target_horizon=20,
        df_attention=df_attention,
        df_jobs=df_jobs,
        df_tnx=df_tnx
    )
    
    model, metrics, best_thresh, df_imp = train_model_and_evaluate(clean_df, feature_cols)
    return metrics, df_imp

# 1. With High-Low Features
metrics_with_hl, imp_with_hl = train_and_get_importance(df_stock)

# 2. Without High-Low Features (Simulate by dropping High/Low columns before feature building)
df_stock_no_hl = df_stock.drop(columns=['High', 'Low'])
metrics_no_hl, imp_no_hl = train_and_get_importance(df_stock_no_hl)

In [ ]:
print("=== Performance Comparison ===")
print(f"Without High-Low Features:")
print(f"  AUC: {metrics_no_hl['auc']:.4f}")
print(f"  F1 Score: {metrics_no_hl['f1']:.4f}")
print(f"  Accuracy: {metrics_no_hl['accuracy']:.4f}")

print(f"\nWith High-Low Features:")
print(f"  AUC: {metrics_with_hl['auc']:.4f}")
print(f"  F1 Score: {metrics_with_hl['f1']:.4f}")
print(f"  Accuracy: {metrics_with_hl['accuracy']:.4f}")

auc_diff = metrics_with_hl['auc'] - metrics_no_hl['auc']
print(f"\nAUC Improvement: {auc_diff:+.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

top_n = 15

# Plot Without H/L
ax1.barh(imp_no_hl['Feature'][:top_n][::-1], imp_no_hl['Gain'][:top_n][::-1], color='lightgray')
ax1.set_title('Feature Importance (Without High/Low Indicators)')
ax1.set_xlabel('Gain')

# Plot With H/L
# Highlight new features
hl_features = [f for f in imp_with_hl['Feature'] if any(x in f for x in ['ATR', 'HL_Range', 'Price_Position', 'Close_Position', 'Close_to_High', 'Up_Down_Ratio', 'Stoch', 'ADX', 'Keltner'])]

colors = ['royalblue' if feat in hl_features else 'lightgray' for feat in imp_with_hl['Feature'][:top_n][::-1]]
ax2.barh(imp_with_hl['Feature'][:top_n][::-1], imp_with_hl['Gain'][:top_n][::-1], color=colors)
ax2.set_title('Feature Importance (With High/Low Indicators)')
ax2.set_xlabel('Gain')

plt.tight_layout()
plt.show()

In [ ]:
print("\n=== Top 10 Features with High-Low Indicators ===")
print(imp_with_hl.head(10))